# Alexa Review Sentiment Classifier

A portfolio rebuild of a class sentiment classifier on the Amazon Alexa reviews
dataset. The target `feedback` (1 positive, 0 negative) is predicted from the free
text in `verified_reviews`.

The point of this project is modeling judgment and label skepticism. The `feedback`
label is a hard threshold on the star `rating`, so "sentiment" here is really a
thresholded star count. The interesting cases are where the text and the stars
disagree. This notebook distrusts the label, characterizes it, then builds around it.

The notebook is staged. This first pass sets up the narrative and one leak-free
`Pipeline`. Rigorous model selection (Phase 2), threshold and PR analysis (Phase 3),
and the fairness and coefficient work (Phase 4) deepen the later sections.

## Imports

Everything imported here, once. Nothing is imported further down.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

RANDOM_STATE = 42
DATA_PATH = "../data/raw/amazon_alexa.tsv"
pd.set_option("display.max_colwidth", 120)

## Data and label

Goal: understand what the label actually is before modeling anything. We load the
raw reviews, confirm how `feedback` relates to `rating`, and measure the class
imbalance. If `feedback` is a deterministic function of `rating`, then the target is
a thresholded star rating rather than an independent human judgment, and that frames
the rest of the project.

In [2]:
df = pd.read_csv(DATA_PATH, sep="\t")
df.columns = [c.strip().lstrip("﻿") for c in df.columns]
print("shape:", df.shape)
df.head()

shape: (3150, 5)


,rating,date,variation,verified_reviews,feedback
0,5,31-Jul-18,Charcoal Fabric,Love my Echo!,1
1,5,31-Jul-18,Charcoal Fabric,Loved it!,1
2,4,31-Jul-18,Walnut Finish,"Sometimes while playing a game, you can answer a question correctly but Alexa says you got it wrong and answers the ...",1
3,5,31-Jul-18,Charcoal Fabric,"I have had a lot of fun with this thing. My 4 yr old learns about dinosaurs, i control the lights and play games lik...",1
4,5,31-Jul-18,Charcoal Fabric,Music,1


Cross-tabulating `feedback` against `rating` shows the mapping directly.

In [3]:
crosstab = pd.crosstab(df["rating"], df["feedback"], margins=True)
crosstab

feedback,0,1,All
rating,,,
1,161,0,161
2,96,0,96
3,0,152,152
4,0,455,455
5,0,2286,2286
All,257,2893,3150


Every rating maps to exactly one feedback value: ratings 1 and 2 give `feedback = 0`,
ratings 3, 4, and 5 give `feedback = 1`. We verify this holds for every row, so the
label carries no information beyond the star threshold.

In [4]:
deterministic = (df.groupby("rating")["feedback"].nunique() == 1).all()
threshold_rule = ((df["rating"].isin([1, 2])) == (df["feedback"] == 0)).all()
print("feedback is a deterministic function of rating:", bool(deterministic))
print("rule 'rating in {1,2} -> 0, else 1' holds for all rows:", bool(threshold_rule))

feedback is a deterministic function of rating: True
rule 'rating in {1,2} -> 0, else 1' holds for all rows: True


The class balance is heavily positive. This is why accuracy will be the wrong headline
metric later: a model that predicts "positive" for everything already scores in the
low nineties.

In [5]:
balance = df["feedback"].value_counts(normalize=True).sort_index()
print("negative (0) share: %.4f" % balance.loc[0])
print("positive (1) share: %.4f" % balance.loc[1])
print("negative count:", int((df["feedback"] == 0).sum()))

fig = px.histogram(
    df, x="feedback",
    title="Label balance: feedback is ~92%% positive",
    labels={"feedback": "feedback (0 = negative, 1 = positive)"},
)
fig.update_layout(bargap=0.2)
fig

negative (0) share: 0.0816
positive (1) share: 0.9184
negative count: 257


## Preprocessing

Goal: get to a clean text/label split without leaking anything. We drop rows with no
usable review text, then hold out a stratified test set. All text vectorization
happens inside the modeling `Pipeline`, so the vectorizer never sees the test data.

`stratify` preserves the roughly 8%% negative prevalence in both splits. With a class
this rare, an unstratified random draw could easily hand the test set too few (or too
many) negatives and distort every minority-class estimate. The cost is minor: the
split is slightly less "random," which is an acceptable trade for stable estimates on
the class we care about.

In [6]:
# Some reviews are NaN or blank; they carry no text signal, so drop them.
text = df["verified_reviews"].astype(str).str.strip()
usable = text.ne("") & df["verified_reviews"].notna()
print("dropped rows with no review text:", int((~usable).sum()))

data = df.loc[usable].copy()
X = data["verified_reviews"].astype(str)
y = data["feedback"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("train:", X_train.shape[0], " test:", X_test.shape[0])
print("train negative share: %.4f" % (1 - y_train.mean()))
print("test  negative share: %.4f" % (1 - y_test.mean()))

dropped rows with no review text: 80
train: 2456  test: 614
train negative share: 0.0774
test  negative share: 0.0765


## Model selection

Goal: establish one correct, leak-free baseline pipeline and read its honest result.
The full comparison (L1 vs L2 penalties on the `saga` solver, a stated `C` grid,
counts vs TF-IDF, unigram vs bigram, and a random-forest contender tuned on
`max_depth`) belongs to Phase 2. Here we just wire the pieces together correctly.

The encoding is a bag-of-words / term-frequency representation: `CountVectorizer`
turns each review into counts of the words it contains. The classifier is logistic
regression with an L2 penalty. Stating the penalty matters: a `C` value means nothing
without saying whether it is L1 or L2.

The core overfitting risk is that the raw vocabulary (features, M) is much larger than
the row count (N). We show this directly: the unpruned bag-of-words vocabulary exceeds
the number of training rows. That is why the model must be regularized, not because
"logistic regression handles high dimensions well." We then prune very rare terms with
`min_df=2` (dropping words that appear in only one review), which shrinks M and cuts
noise, instead of using the built-in `english` stop-word list. That list was built for
computer-science text and drops words like "computer," so it is not appropriate here.

In [7]:
# Raw feature space vs training rows: M (raw) >> N is the overfitting risk.
raw_vocab = len(CountVectorizer(min_df=1).fit(X_train).vocabulary_)
print("raw vocabulary (min_df=1), M_raw: %d   training rows, N: %d" % (raw_vocab, X_train.shape[0]))
print("M_raw >> N:", raw_vocab > X_train.shape[0])

# One Pipeline. The vectorizer is fit only on training data, inside .fit().
baseline = Pipeline([
    ("bow", CountVectorizer(min_df=2)),         # bag-of-words / term-frequency counts
    ("clf", LogisticRegression(
        penalty="l2", C=1.0, solver="liblinear", max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])
baseline.fit(X_train, y_train)

pruned_vocab = len(baseline.named_steps["bow"].vocabulary_)
print("after min_df=2 pruning, M: %d (regularization handles the rest)" % pruned_vocab)

raw vocabulary (min_df=1), M_raw: 3669   training rows, N: 2456
M_raw >> N: True
after min_df=2 pruning, M: 2206 (regularization handles the rest)


## Evaluation

Goal: judge the model against the do-nothing baseline, minority class first. We never
lead with accuracy. We report the `DummyClassifier(most_frequent)` floor, then the
negative-class precision, recall, and specificity for the pipeline, with a confusion
matrix. Precision-recall curves, PR-AUC, and threshold tuning come in Phase 3; this
section reads the default-threshold result honestly so we know what needs improving.

In [8]:
def negative_class_report(name, y_true, y_pred):
    # Negative class (0) is the minority and the class of interest.
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "neg_precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "neg_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "specificity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "neg_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
    }

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
rows = [
    negative_class_report("DummyClassifier (most_frequent)", y_test, dummy.predict(X_test)),
    negative_class_report("LogReg L2 (baseline, threshold 0.5)", y_test, baseline.predict(X_test)),
]
report = pd.DataFrame(rows).set_index("model").round(4)
report

,accuracy,neg_precision,neg_recall,specificity,neg_f1
model,,,,,
DummyClassifier (most_frequent),0.9235,0.0000,0.0000,1.0000,0.0000
"LogReg L2 (baseline, threshold 0.5)",0.9528,0.8462,0.4681,0.9929,0.6027


Specificity and precision answer different questions and both matter here. Specificity
is "how often are positive reviews wrongly flagged as negative"; precision is "how
trustworthy is a negative flag." The confusion matrix below makes the trade concrete.

In [9]:
cm = confusion_matrix(y_test, baseline.predict(X_test), labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["true negative (0)", "true positive (1)"],
    columns=["pred negative (0)", "pred positive (1)"],
)
cm_df

,pred negative (0),pred positive (1)
true negative (0),22,25
true positive (1),4,563


Reading the baseline honestly: the pipeline clears the majority-class accuracy floor,
but its negative-class recall is modest, meaning it still misses a large share of the
genuinely negative reviews at the default 0.5 threshold. That gap, not the headline
accuracy, is the problem the later phases attack with class weighting, a proper penalty
and encoding comparison (Phase 2), and threshold tuning against a stated recall target
(Phase 3).

## Fairness

Goal (delivered in Phase 4): treat fairness as separation across device `variation`
groups, that is, check whether negative-class recall and the false-positive rate are
roughly equal across variations rather than only checking overall accuracy. This
section will report the spread across groups and name the criterion explicitly. It is
stubbed here so the narrative structure is in place; the computation lands in Phase 4.

## Further analysis

Goal (delivered in Phase 4): interpret the model rather than just score it. Using the
sparse L1 coefficients from the Phase 2 winner, we will surface the words that push a
review toward the negative class and argue why an L1 penalty serves the explainability
goal better than L2. We will also plot the predicted negative rate by device
`variation`. Stubbed here, delivered in Phase 4.